# Task 5 - Model Evaluation: Cardiovascular Disease Dataset

**Dataset:** [Kaggle - Cardiovascular Disease Dataset](https://www.kaggle.com/datasets/sulianova/cardiovascular-disease-dataset)

**Legend:**
- [BOTH] = apply to every model
- [CLASS] = classification only  
- [REG] = regression only

---

## 0. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
import os

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    mean_squared_error, r2_score
)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    AdaBoostClassifier, AdaBoostRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries imported successfully.')

## 1. Load the Dataset

In [ ]:
# The Kaggle dataset ships as cardio_train.csv (semicolon-separated).
# In this project it may also be stored as cardio_train.csv.xlsx.

file_xlsx = 'cardio_train.csv.xlsx'
file_csv  = 'cardio_train.csv'

if os.path.exists(file_xlsx):
    df = pd.read_excel(file_xlsx)
    print(f'Loaded from Excel: {file_xlsx}')
elif os.path.exists(file_csv):
    df = pd.read_csv(file_csv, sep=';')
    print(f'Loaded from CSV: {file_csv}')
else:
    raise FileNotFoundError(
        'Dataset not found. Download cardio_train.csv from:\n'
        'https://www.kaggle.com/datasets/sulianova/cardiovascular-disease-dataset\n'
        'and place it in the ML-project folder.'
    )

print(f'Shape: {df.shape}')
df.head()

## 2. Quick EDA and Preprocessing

In [ ]:
# Drop id column if present
if 'id' in df.columns:
    df.drop(columns=['id'], inplace=True)

print('Columns:', df.columns.tolist())
print('\nMissing values:\n', df.isnull().sum())
print('\nClass distribution (cardio):\n', df['cardio'].value_counts())

# Remove obvious outliers (blood pressure outliers common in this dataset)
df = df[(df['ap_hi'] >= 80)  & (df['ap_hi'] <= 240)]
df = df[(df['ap_lo'] >= 40)  & (df['ap_lo'] <= 160)]
df = df[(df['height'] >= 130) & (df['height'] <= 220)]
df = df[(df['weight'] >= 30)  & (df['weight'] <= 200)]

# Feature engineering: BMI
df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)

# Age in years (dataset stores age in days)
if df['age'].max() > 200:
    df['age'] = (df['age'] / 365).round(1)

print(f'\nCleaned shape: {df.shape}')

## 3. Train / Test Split and Scaling

In [ ]:
X = df.drop(columns=['cardio'])
y = df['cardio']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Training set : {X_train_sc.shape}')
print(f'Test set     : {X_test_sc.shape}')

---
# SECTION 1 - Model Evaluation [CLASS]

In [ ]:
def eval_classifier(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    return model, {
        'Model':      name,
        'Accuracy':   round(accuracy_score(y_te, y_pred)*100, 2),
        'Precision':  round(precision_score(y_te, y_pred)*100, 2),
        'Recall':     round(recall_score(y_te, y_pred)*100, 2),
        'F1-Score':   round(f1_score(y_te, y_pred)*100, 2),
        'Train_Score': round(model.score(X_tr, y_tr)*100, 2),
        'Test_Score':  round(model.score(X_te, y_te)*100, 2),
    }

clf_results = []
clf_models  = {}

classifiers = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'       : DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'AdaBoost'            : AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
}

for name, clf in classifiers.items():
    model, m = eval_classifier(name, clf, X_train_sc, y_train, X_test_sc, y_test)
    clf_models[name] = model
    clf_results.append(m)
    print(f'{name}: Acc={m["Accuracy"]}%  F1={m["F1-Score"]}%')

clf_df = pd.DataFrame(clf_results)
print('\n--- Classification Results ---')
clf_df

---
# SECTION 2 - Check Overfitting / Underfitting [BOTH]

In [ ]:
print('Model                    | Train  | Test   | Gap    | Diagnosis')
print('-' * 70)
for row in clf_results:
    gap = round(row['Train_Score'] - row['Test_Score'], 2)
    if gap > 10:
        diagnosis = 'Overfitting'
    elif row['Train_Score'] < 70 and row['Test_Score'] < 70:
        diagnosis = 'Underfitting'
    else:
        diagnosis = 'Good Fit'
    print(f'{row["Model"]:<25}| {row["Train_Score"]:>5}% | {row["Test_Score"]:>5}% | {gap:>5}% | {diagnosis}')

---
# SECTION 3 - 5-Fold Cross-Validation [BOTH]

In [ ]:
print('Model                    | Avg CV Score | Std (Spread) | Stability')
print('=' * 70)

for name, clf in classifiers.items():
    scores = cross_val_score(clf, X_train_sc, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    avg    = scores.mean()
    spread = scores.std()
    if spread < 0.01:
        stability = 'Stable'
    elif spread < 0.02:
        stability = 'Moderate'
    else:
        stability = 'Unstable'
    print(f'{name:<25}| {avg*100:>11.2f}% | {spread:>12.5f} | {stability}')
    print(f'  Fold scores: {[round(s, 4) for s in scores]}')
    print('-' * 70)

---
# SECTION 4A - Compare Classification Models Table

In [ ]:
compare_clf = clf_df[['Model','Accuracy','Precision','Recall','F1-Score']].copy()
compare_clf = compare_clf.sort_values('F1-Score', ascending=False).reset_index(drop=True)
print('--- Classification Model Comparison Table ---')
print(compare_clf.to_string(index=False))
best_clf_name = compare_clf.iloc[0]['Model']
print(f'\nBest Classification Model: {best_clf_name}')

---
# SECTION 4B - Regression Models [REG]

In [ ]:
def eval_regressor(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    rss  = round(float(np.sum((y_te - y_pred) ** 2)), 2)
    rmse = round(float(np.sqrt(mean_squared_error(y_te, y_pred))), 4)
    r2   = round(float(r2_score(y_te, y_pred)), 4)
    return model, {
        'Model': name, 'RSS': rss, 'RMSE': rmse, 'R2': r2,
        'Train_Score': round(model.score(X_tr, y_tr), 4),
        'Test_Score':  round(model.score(X_te, y_te), 4),
    }

reg_results = []
reg_models  = {}

regressors = {
    'Linear Regression'      : LinearRegression(),
    'Ridge Regression'       : Ridge(alpha=1.0),
    'Lasso Regression'       : Lasso(alpha=0.001, max_iter=5000),
    'Random Forest Reg.'     : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting Reg.' : GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
}

for name, reg in regressors.items():
    model, m = eval_regressor(name, reg, X_train_sc, y_train, X_test_sc, y_test)
    reg_models[name] = model
    reg_results.append(m)

reg_df = pd.DataFrame(reg_results).sort_values('R2', ascending=False).reset_index(drop=True)
print('--- Regression Model Comparison Table ---')
print(reg_df[['Model','RSS','RMSE','R2']].to_string(index=False))
best_reg_name = reg_df.iloc[0]['Model']
print(f'\nBest Regression Model: {best_reg_name}')

---
# SECTION 5 - Hyperparameter Tuning [BOTH]

In [ ]:
# --- Classification: GridSearchCV on Gradient Boosting ---
print('Running GridSearchCV for GradientBoostingClassifier ...')

param_grid_clf = {
    'n_estimators' : [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth'    : [3, 5],
    'subsample'    : [0.8, 1.0],
}

gb_clf = GradientBoostingClassifier(random_state=42)
grid_clf = GridSearchCV(gb_clf, param_grid_clf, cv=5, scoring='accuracy', n_jobs=-1, verbose=0)
grid_clf.fit(X_train_sc, y_train)

print(f'\nBest Params (Classification): {grid_clf.best_params_}')
print(f'Best CV Score              : {grid_clf.best_score_*100:.2f}%')

best_clf = grid_clf.best_estimator_
y_pred_tuned_clf = best_clf.predict(X_test_sc)
tuned_acc = accuracy_score(y_test, y_pred_tuned_clf) * 100
print(f'Tuned Test Accuracy        : {tuned_acc:.2f}%')
base_acc = clf_df[clf_df['Model']=='Gradient Boosting']['Accuracy'].values[0]
print('Score IMPROVED after tuning!' if tuned_acc > base_acc else 'Score comparable to baseline.')

In [ ]:
# --- Regression: RandomizedSearchCV on Gradient Boosting Reg. ---
from scipy.stats import randint, uniform

print('Running RandomizedSearchCV for GradientBoostingRegressor ...')

param_dist_reg = {
    'n_estimators' : randint(100, 300),
    'learning_rate': uniform(0.05, 0.15),
    'max_depth'    : randint(3, 6),
    'subsample'    : uniform(0.7, 0.3),
}

gb_reg = GradientBoostingRegressor(random_state=42)
rand_reg = RandomizedSearchCV(gb_reg, param_dist_reg, n_iter=20, cv=5,
                               scoring='r2', n_jobs=-1, random_state=42, verbose=0)
rand_reg.fit(X_train_sc, y_train)

print(f'\nBest Params (Regression): {rand_reg.best_params_}')
print(f'Best CV R2             : {rand_reg.best_score_:.4f}')

best_reg = rand_reg.best_estimator_
y_pred_tuned_reg = best_reg.predict(X_test_sc)
tuned_r2 = r2_score(y_test, y_pred_tuned_reg)
print(f'Tuned Test R2          : {tuned_r2:.4f}')
base_r2 = reg_df[reg_df['Model']=='Gradient Boosting Reg.']['R2'].values[0]
print('R2 IMPROVED after tuning!' if tuned_r2 > base_r2 else 'R2 comparable to baseline.')

---
# SECTION 6 - Try Advanced Models [BOTH] + Optional Gradient Boosting

In [ ]:
# --- Random Forest (Bagging) [BOTH] ---
rf_adv = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_adv.fit(X_train_sc, y_train)
rf_acc = accuracy_score(y_test, rf_adv.predict(X_test_sc)) * 100
print(f'Random Forest (Bagging) Accuracy : {rf_acc:.2f}%')
rf_cv = cross_val_score(rf_adv, X_train_sc, y_train, cv=5, scoring='accuracy', n_jobs=-1)
print(f'  5-Fold CV: avg={rf_cv.mean()*100:.2f}%  spread={rf_cv.std():.5f}')

# --- AdaBoost [BOTH] ---
ada_adv = AdaBoostClassifier(n_estimators=200, learning_rate=0.5, random_state=42)
ada_adv.fit(X_train_sc, y_train)
ada_acc = accuracy_score(y_test, ada_adv.predict(X_test_sc)) * 100
print(f'\nAdaBoost Accuracy                : {ada_acc:.2f}%')
ada_cv = cross_val_score(ada_adv, X_train_sc, y_train, cv=5, scoring='accuracy', n_jobs=-1)
print(f'  5-Fold CV: avg={ada_cv.mean()*100:.2f}%  spread={ada_cv.std():.5f}')

# --- Gradient Boosting [BOTH] (Optional) ---
gb_adv = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
gb_adv.fit(X_train_sc, y_train)
gb_acc = accuracy_score(y_test, gb_adv.predict(X_test_sc)) * 100
print(f'\nGradient Boosting (Optional) Acc : {gb_acc:.2f}%')
gb_cv = cross_val_score(gb_adv, X_train_sc, y_train, cv=5, scoring='accuracy', n_jobs=-1)
print(f'  5-Fold CV: avg={gb_cv.mean()*100:.2f}%  spread={gb_cv.std():.5f}')

---
# Save Best Model as .pkl

In [ ]:
pkl_path = 'best_cardio_model.pkl'
with open(pkl_path, 'wb') as f:
    pickle.dump({
        'model'   : best_clf,
        'scaler'  : scaler,
        'params'  : grid_clf.best_params_,
        'features': list(X.columns),
        'accuracy': tuned_acc,
    }, f)

print(f'Best model saved to: {pkl_path}')

# Verify load
with open(pkl_path, 'rb') as f:
    loaded = pickle.load(f)
print(f'Model type : {type(loaded["model"]).__name__}')
print(f'Accuracy   : {loaded["accuracy"]:.2f}%')
print(f'Best params: {loaded["params"]}')

## Summary

| Section | Task | Status |
|---------|------|--------|
| 1 | Model Evaluation ? Accuracy / Precision / Recall / F1 (CLASS), RSS / RMSE / R2 (REG) | Done |
| 2 | Overfitting / Underfitting Check (BOTH) | Done |
| 3 | 5-Fold Cross-Validation (BOTH) | Done |
| 4 | Compare All Models ? Classification & Regression tables (BOTH) | Done |
| 5 | Hyperparameter Tuning ? GridSearchCV + RandomizedSearchCV (BOTH) | Done |
| 6 | Advanced Models ? Random Forest, AdaBoost, Gradient Boosting (BOTH + Optional) | Done |
| PKL | best_cardio_model.pkl saved | Done |
